# 🏗️ Fase 3: Data Consolidation (Merge Master)

Este cuaderno ejecuta la estrategia de cruce (Merge) diseñada en la Arquitectura End-to-End, transformando los módulos individuales (`RECH6`, `RECH1`, `RECH0`, `RECH23`) en un único Dataset Analítico Maestro a nivel de **Niño**.

## Reglas Epidemiológicas Aplicadas:
1. **Target Individual:** 1 Fila = 1 Niño (Menor de 5 años medido en `RECH6`).
2. **Filtro de Residencia:** Solo se incluyen niños que durmieron en el hogar la noche anterior (`HV103 == 1`).


In [1]:
import pandas as pd
from pathlib import Path

# Configuraciones
pd.set_option('display.max_columns', None)
interim_dir = Path('../../data/interim')


In [2]:
# 1. Cargar las 4 tablas base
print("⏳ Cargando tablas interim...")
df_rech6 = pd.read_parquet(interim_dir / 'rech6_cleaned.parquet')
df_rech1 = pd.read_parquet(interim_dir / 'rech1_cleaned.parquet')
df_rech0 = pd.read_parquet(interim_dir / 'rech0_cleaned.parquet')
df_rech23 = pd.read_parquet(interim_dir / 'rech23_cleaned.parquet')

print(f"[OK] RECH6 (Tronco): {len(df_rech6):,} filas")
print(f"[OK] RECH1 (Demografía): {len(df_rech1):,} filas")
print(f"[OK] RECH0 (Entrevista): {len(df_rech0):,} filas")
print(f"[OK] RECH23 (Hogar): {len(df_rech23):,} filas")


⏳ Cargando tablas interim...
[OK] RECH6 (Tronco): 294,109 filas
[OK] RECH1 (Demografía): 2,291,232 filas
[OK] RECH0 (Entrevista): 569,283 filas
[OK] RECH23 (Hogar): 607,287 filas


### 🛡️ Filtro Epidemiológico en RECH1 (`HV103 == 1`)

In [3]:
# Aplicar regla del INEI: Solo miembros que durmieron en casa anoche
# HV103 puede tener múltiples tipos de datos al agrupar 18 años (float, int, string)
total_original = len(df_rech1)
valid_values = [1, 1.0, '1', '1.0', 'Sí', 'Yes', 'si']
df_rech1 = df_rech1[df_rech1['HV103'].isin(valid_values)].copy()

print(f"Se filtraron {total_original - len(df_rech1):,} personas que NO durmieron en casa.")
print(f"Quedan {len(df_rech1):,} residentes habituales válidos.")


Se filtraron 114,042 personas que NO durmieron en casa.
Quedan 2,177,190 residentes habituales válidos.


### 🛠️ Función Auxiliar para evitar columnas duplicadas

In [4]:
def drop_overlapping_columns(left_df, right_df, join_keys_right):
    """Elimina columnas del dataframe derecho que ya existen en el izquierdo, excepto las llaves de cruce."""
    overlap = set(right_df.columns).intersection(set(left_df.columns))
    cols_to_drop = [c for c in overlap if c not in join_keys_right]
    return right_df.drop(columns=cols_to_drop)


### 🔄 Cruce 1: Base (RECH6) + Demografía (RECH1)

In [5]:
# Left Join por Niño
# El INEI recicla los HHID, por lo tanto la llave primaria obligatoria siempre debe incluir el año (`year`)
join_keys_child = ['year', 'HHID', 'HVIDX']
df_rech1_clean = drop_overlapping_columns(df_rech6, df_rech1, join_keys_child)

master = pd.merge(
    df_rech6, 
    df_rech1_clean, 
    left_on=['year', 'HHID', 'HC0'], 
    right_on=['year', 'HHID', 'HVIDX'], 
    how='left'
)

print(f"Filas resultantes: {len(master):,} (Debería ser exactamente igual a {len(df_rech6):,})")


Filas resultantes: 294,109 (Debería ser exactamente igual a 294,109)


### 🔄 Cruce 2 y 3: Entorno del Hogar (RECH0 y RECH23)

In [6]:
# Left Join por Hogar
join_keys_hh = ['year', 'HHID']

df_rech0_clean = drop_overlapping_columns(master, df_rech0, join_keys_hh)
master = pd.merge(master, df_rech0_clean, on=join_keys_hh, how='left')

df_rech23_clean = drop_overlapping_columns(master, df_rech23, join_keys_hh)
master = pd.merge(master, df_rech23_clean, on=join_keys_hh, how='left')

print(f"Filas resultantes: {len(master):,} (Debería ser exactamente igual a {len(df_rech6):,})")
print(f"Columnas resultantes: {master.shape[1]}")


Filas resultantes: 294,109 (Debería ser exactamente igual a 294,109)
Columnas resultantes: 183


### 🔄 Cruce 4: Historia de Embarazo y Lactancia (REC41) a través de REC21 (Tabla Puente)

In [7]:
# Cargar REC21 (Puente) usando el motor de ingesta (Historia completa multi-año)
from mnp.ingestion.loader import load_endes

print("⏳ Cargando REC21 (Historia de Nacimientos) como Puente...")
# Cargamos toda la historia de REC21 inyectando el AÑO para evitar el producto cartesiano
dict_rec21 = load_endes(module='birth_history', record='rec21', usecols=['CASEID', 'BIDX', 'B16'])
dfs = []
for y, df in dict_rec21.items():
    df['year'] = int(y)
    dfs.append(df)
df_rec21 = pd.concat(dfs, ignore_index=True)

# Castear llaves a string (mismo trato que la fase de limpieza)
df_rec21['CASEID'] = df_rec21['CASEID'].astype(str).str.strip()

# Filtrar nulos (B16 es 0 o nulo si el hijo no reside en el hogar o falleció)
df_rec21 = df_rec21.dropna(subset=['B16'])
df_rec21 = df_rec21[df_rec21['B16'].astype(float) > 0].copy()

# Castear BIDX y B16 a string limpio (eliminando .0)
df_rec21['BIDX'] = df_rec21['BIDX'].astype(float).astype(int).astype(str)
df_rec21['B16'] = df_rec21['B16'].astype(float).astype(int).astype(str)

# Cargar REC41 (Features limpios)
print("⏳ Cargando REC41 (Features)...")
df_rec41 = pd.read_parquet(interim_dir / 'rec41_features.parquet')

# Castear MIDX de REC41 a string limpio para que cruce con BIDX
df_rec41['MIDX'] = df_rec41['MIDX'].astype(float).astype(int).astype(str)

# 1. Cruzar REC41 con REC21 para inyectar B16 (El HVIDX/HC0 universal del niño)
print("🔗 Cruzando REC41 con el Puente REC21...")
df_rec41_bridged = pd.merge(
    df_rec41,
    df_rec21,
    left_on=['year', 'CASEID', 'MIDX'],
    right_on=['year', 'CASEID', 'BIDX'],
    how='inner' # Nos quedamos solo con los que tienen B16 válido (viven en el hogar)
)

# 2. Reconstruir HHID desde CASEID para el cruce maestro
# CASEID en ENDES siempre termina con la línea de la madre (3 caracteres). El resto es HHID.
df_rec41_bridged['extracted_HHID'] = df_rec41_bridged['CASEID'].str[:-3].str.strip()

# 3. Cruzar al Master!
# HHID = extracted_HHID  Y  HC0 = B16
print("🔗 Inyectando REC41 al Master...")
df_rec41_clean = drop_overlapping_columns(master, df_rec41_bridged, ['year', 'extracted_HHID', 'B16'])

master = pd.merge(
    master,
    df_rec41_clean,
    left_on=['year', 'HHID', 'HC0'],
    right_on=['year', 'extracted_HHID', 'B16'],
    how='left'
)

print(f"Filas resultantes: {len(master):,} (Debería ser exactamente igual a 294,109)")
print(f"Columnas resultantes: {master.shape[1]}")

2026-06-16 18:50:23.241 | INFO     | mnp.config:<module>:11 - PROJ_ROOT path is: /Users/abelguevarah/Desktop/invs/malnutrition-research


⏳ Cargando REC21 (Historia de Nacimientos) como Puente...
2026-06-16 18:50:23.387 | INFO     | mnp.ingestion.loader:_load_multiple_years:285 - 🔎 Detectada historia de 23 años.
2026-06-16 18:50:23.435 | INFO     | mnp.ingestion.loader:load_endes:380 - ✓ Cargando registro: REC21_2024.sav desde Modulo1632 [Alias: rec21]
2026-06-16 18:50:23.667 | INFO     | mnp.ingestion.loader:load_endes:380 - ✓ Cargando registro: REC21_2023.sav desde Modulo1632 [Alias: rec21]
2026-06-16 18:50:23.856 | INFO     | mnp.ingestion.loader:load_endes:380 - ✓ Cargando registro: REC21.sav desde Modulo1632 [Alias: rec21]
2026-06-16 18:50:24.049 | INFO     | mnp.ingestion.loader:load_endes:380 - ✓ Cargando registro: REC21.SAV desde Modulo1632 [Alias: rec21]
2026-06-16 18:50:24.245 | INFO     | mnp.ingestion.loader:load_endes:380 - ✓ Cargando registro: REC21.sav desde Modulo1632 [Alias: rec21]
2026-06-16 18:50:24.570 | INFO     | mnp.ingestion.loader:load_endes:380 - ✓ Cargando registro: REC21.sav desde Modulo67 [Al

### 🍼 Cálculo de KPI 5 (Lactancia Materna Exclusiva)

In [8]:
# Regla Oficial del MINSA/INEI (Ficha Técnica KPI 5)
# 1. Niño menor de 6 meses (HC1 < 6)
# 2. Aún le está dando pecho (M4 == 95.0)
# (Nota: La lógica oficial también valida REC42 para líquidos. Como proxy de alta pureza 
# usamos M55A para asegurar exclusividad en primeros días o simplemente la intención principal).

master['kpi5_lactancia_exclusiva'] = ((master['M4'] == 95.0) & (master['HC1'] < 6)).astype(int)

# IMPORTANTE: Arreglar M4 para XGBoost
# Si dejamos el código 95.0, XGBoost pensará que el niño lactó 95 meses.
# Matemáticamente, si el niño "aún lacta" (95.0), los meses reales de lactancia 
# son idénticos a su edad actual en meses (HC1).
mask_aun_lacta = master['M4'] == 95.0
master.loc[mask_aun_lacta, 'M4'] = master.loc[mask_aun_lacta, 'HC1']

print("KPI 5 Calculado y M4 matematizado.")
print(master['kpi5_lactancia_exclusiva'].value_counts(dropna=False))

KPI 5 Calculado y M4 matematizado.
kpi5_lactancia_exclusiva
0    294109
Name: count, dtype: int64


In [9]:
# Vamos a demostrar matemáticamente que el niño que unimos desde REC41 es EXACTAMENTE el mismo niño de RECH6.
# Para esto, traeremos la variable Sexo (B4) y Año de Nacimiento (B2) desde la historia clínica REC21.
# Y las compararemos contra el Sexo (HC27) y Año de Nacimiento (HC31) de la Antropometría RECH6.

print("⏳ Cargando datos biológicos de REC21 para auditoría...")
dict_audit = load_endes(module='birth_history', record='rec21', usecols=['CASEID', 'BIDX', 'B2', 'B4'])
dfs_audit = []
for y, df in dict_audit.items():
    df['year'] = int(y)
    dfs_audit.append(df)
df_audit = pd.concat(dfs_audit, ignore_index=True)

# Limpieza de llaves
df_audit['CASEID'] = df_audit['CASEID'].astype(str).str.strip()
df_audit['BIDX'] = df_audit['BIDX'].astype(float).astype(int).astype(str)

# Pegamos esta info de auditoría al master (usando CASEID y MIDX que sobrevivieron al cruce anterior)
master_audit = pd.merge(
    master,
    df_audit,
    left_on=['year', 'CASEID', 'MIDX'],
    right_on=['year', 'CASEID', 'BIDX'],
    how='inner'
)

# Comparativa 1: SEXO
# HC27 ya está decodificado como string ("Hombre", "Mujer") en el Master.
# B4 viene crudo como numérico (1=Hombre, 2=Mujer)
map_sexo = {'Hombre': 1, 'Masculino': 1, 'Mujer': 2, 'Femenino': 2}
master_audit['HC27_num'] = master_audit['HC27'].str.strip().str.capitalize().map(map_sexo)

sexo_match = (master_audit['HC27_num'] == master_audit['B4'].astype(float)).mean() * 100

# Comparativa 2: AÑO DE NACIMIENTO
year_match = (master_audit['HC31'].astype(float) == master_audit['B2'].astype(float)).mean() * 100

print(f"✅ Coincidencia de Sexo Biológico entre módulos: {sexo_match:.2f}%")
print(f"✅ Coincidencia de Año de Nacimiento entre módulos: {year_match:.2f}%")
print("\nSi ambos números son 100% (o 99.9%), es la prueba absoluta de que el cruce MIDX -> HC0 fue perfecto.")


⏳ Cargando datos biológicos de REC21 para auditoría...
2026-06-16 18:50:29.588 | INFO     | mnp.ingestion.loader:_load_multiple_years:285 - 🔎 Detectada historia de 23 años.
2026-06-16 18:50:29.591 | INFO     | mnp.ingestion.loader:load_endes:380 - ✓ Cargando registro: REC21_2024.sav desde Modulo1632 [Alias: rec21]
2026-06-16 18:50:29.816 | INFO     | mnp.ingestion.loader:load_endes:380 - ✓ Cargando registro: REC21_2023.sav desde Modulo1632 [Alias: rec21]
2026-06-16 18:50:30.040 | INFO     | mnp.ingestion.loader:load_endes:380 - ✓ Cargando registro: REC21.sav desde Modulo1632 [Alias: rec21]
2026-06-16 18:50:30.272 | INFO     | mnp.ingestion.loader:load_endes:380 - ✓ Cargando registro: REC21.SAV desde Modulo1632 [Alias: rec21]
2026-06-16 18:50:30.500 | INFO     | mnp.ingestion.loader:load_endes:380 - ✓ Cargando registro: REC21.sav desde Modulo1632 [Alias: rec21]
2026-06-16 18:50:30.715 | INFO     | mnp.ingestion.loader:load_endes:380 - ✓ Cargando registro: REC21.sav desde Modulo67 [Alias

### 📅 Resumen de Casos por Año

In [10]:
# Distribución de niños evaluados a lo largo de los 18 años
resumen_year = master['year'].value_counts().sort_index().reset_index()
resumen_year.columns = ['Año', 'Niños Evaluados']
resumen_year['% del Total'] = (resumen_year['Niños Evaluados'] / len(master) * 100).round(1).astype(str) + '%'

display(resumen_year)


,Año,Niños Evaluados,% del Total
0,2007,5666,1.9%
1,2008,10747,3.7%
2,2009,10704,3.6%
3,2010,9812,3.3%
4,2011,9582,3.3%
5,2012,10231,3.5%
6,2013,9574,3.3%
7,2014,10222,3.5%
8,2015,25527,8.7%
9,2016,22682,7.7%


### 🧹 Limpieza Final: Poda de Columnas Muertas (>70% Nulos)

In [11]:
# Cuando unimos 20 años de encuestas, es normal que aparezcan "Columnas Fantasma".
# Son preguntas que el INEI hizo en un solo año (ej. 2008) y nunca más volvió a preguntar, 
# o preguntas que solo aplican a adultos (ej. Estado Civil HV115 en niños de 5 años).

# 1. Calcular porcentaje de nulos
null_pct = (master.isnull().mean() * 100).sort_values(ascending=False)

# 2. Identificar columnas con más del 70% de nulos
bad_columns = null_pct[null_pct > 70].index.tolist()

print(f"🗑️ Se detectaron {len(bad_columns)} columnas con más del 70% de nulos.")
print("Columnas a eliminar:", bad_columns)

# 3. Eliminar esas columnas del Master (excepto si hay alguna que queramos salvar)
# whitelist = ['alguna_columna_importante']
# bad_columns = [col for col in bad_columns if col not in whitelist]

master = master.drop(columns=bad_columns)

print(f"✅ Limpieza completada. Columnas finales en el Master: {master.shape[1]}")


🗑️ Se detectaron 23 columnas con más del 70% de nulos.
Columnas a eliminar: ['QH13A6D', 'HV115', 'HV118', 'HV243B', 'QH13A6C', 'QH13A6B', 'HV107', 'SH56_Limpió_Baño', 'SH59_Recogen_Basura', 'SH52_Institucion_Agua', 'SH51_Frec_Pago_Agua', 'QH13A6A', 'SH51_Pago_Agua', 'SH48_Basura', 'SH49_Frec_Basura', 'SH52', 'SH43', 'HV236', 'SH51', 'SH56_Frec_Limpieza', 'SH58', 'SH64', 'SH50_Tipo_Basurero']
✅ Limpieza completada. Columnas finales en el Master: 189


In [12]:
# Guardar resultado maestro
output_path = interim_dir / 'master_merged_v2.parquet'
master.to_parquet(output_path, index=False)
print(f"🎉 Dataset maestro guardado con éxito: {output_path}")


🎉 Dataset maestro guardado con éxito: ../../data/interim/master_merged_v2.parquet


### 👀 Vista Previa con Etiquetas (Diccionario INEI)

In [13]:
import sys
sys.path.append('../../')
from mnp.configs.column_labels import LABELS

# Tomamos las primeras 30 filas
vis_df = master.head(30).copy()

# Creamos un MultiIndex para las columnas: (Código_Variable, Etiqueta_Descriptiva)
multi_columns = [(col, LABELS.get(col.upper(), 'Sin Etiqueta')) for col in vis_df.columns]
vis_df.columns = pd.MultiIndex.from_tuples(multi_columns, names=['Variable', 'Descripción'])

display(vis_df)


Variable,HHID,HC0,HC1,HC2,HC3,HC4,HC5,HC6,HC7,HC8,HC9,HC10,HC11,HC12,HC13,HC15,HC16,HC19,HC27,HC30,HC31,HC32,HC33,HC51,HC53,HC55,HC56,HC57,HC60,HC61,HC62,HC63,HC64,HC70,HC71,HC72,HC73,year,HVIDX,HV101,HV102,HV103,HV104,HV105,HV106,HV108,HV109,HV110,HV111,HV112,HV113,HV114,HV117,HV120,HV121,HV122,HV124,HV125,HV126,HV128,HV001,HV002,HV004,HV005,HV008,HV009,HV010,HV012,HV013,HV014,HV015,HV021,HV022,HV024,HV025,HV026,HV035,HV040,ID1,HV002A,CODCCPP,NOMCCPP,UBIGEO,NCONGLOME,LONGITUDX,LATITUDY,HV201,HV204,HV205,HV206,HV207,HV208,HV209,HV210,HV211,HV212,HV213,HV214,HV215,HV216,HV217,HV218,HV219,HV220,HV221,HV225,HV226,HV234,HV237,HV237A,HV237B,HV237C,HV237D,HV237E,HV237F,HV237G,HV237X,HV242,HV243A,HV243C,HV243D,HV244,HV246,HV270,HV271,SHREGION,SHPROVIN,SHDISTRI,SHTOTH,SHSEMES,SH49,SH50,SH63,SH61A,SH61B,SH61C,SH61D,SH61E,SH61J,SH61K,SH61L,SH61M,SH61N,SH61O,SH61P,SH61Q,SH61R,SH61S,SH71,SH76B,SH76C,SH76D,SH77F,SH227,QH227A,QH227B,SH42_Agua_Todo_El_Dia,SH48_Conserva_Agua,SH70_Fuente_Luz,HV240,CASEID,MIDX,M13,M14,M17,M18,M4,M45,M46,M54,M60,M70,M55A,M55B,M55C,M55E,M55F,M55G,M55H,M55I,M55X,M55Z,m34_horas_inicio_lactancia,m19_peso_nacer_kg,m15_lugar_parto_agrupado,BIDX,B16,extracted_HHID,kpi5_lactancia_exclusiva
Descripción,HHID - Identificación del hogar / Cuestionario,HC0 - Número de orden / Índice del listado,HC1 - Edad en meses,HC2 - Peso en kilogramos (1 dec.),HC3 - Altura en centímetros (1 dec.),HC4 - Talla/Edad percentil,HC5 - Talla/Edad Desviación Estándar,HC6 - Talla/Edad porcentaje de la mediana,HC7 - Peso/Edad percentil,HC8 - Peso/Edad Desviación Estándar,HC9 - Peso/Edad porcentaje de la mediana,HC10 - Peso/Talla percentil,HC11 - Peso/Talla Desviación Estándar,HC12 - Peso/Talla porcentaje de la mediana,HC13 - Razón por la cual el niño no se midió,HC15 - El niño se midió acostado o de pie,HC16 - Día de nacimiento del niño,HC19 - Año de la medición,HC27 - Sexo,HC30 - Mes de nacimiento del niño,HC31 - Año de nacimiento del niño,HC32 - Fecha de nacimiento (cmc),HC33 - Fecha de información completa,HC51 - Número de orden del padre o responsable,HC53 - Nivel de hemoglobina (g/dl-1 decimal),HC55 - Resultado de medir (hemoglobina),HC56 - Nivel de hemoglobina ajustado por altitud,HC57 - Nivel de anemia,HC60 - Número de línea de la madre en el hogar,HC61 - Nivel educativo más alto de la madre,HC62 - Año más alto de educación de la madre,HC63 - Intervalo de nacimientos anteriores al niño,HC64 - Número de orden de nacimiento,HC70 - Talla/Edad Desviación Estándar (OMS),HC71 - Peso/Edad Desviación Estándar (OMS),HC72 - Peso/Talla Desviación Estándar (OMS),HC73 - Desviación Estándar del IMC (OMS),Año de Evaluación,HVIDX - Número de orden del individuo,HV101 - Parentesco con jefe de hogar,HV102 - Residente habitual,HV103 - ¿Durmió aquí anoche?,HV104 - Sexo,HV105 - Edad,HV106 - Nivel de estudios más alto,HV108 - Número de años de estudio,HV109 - Nivel educativo alcanzado,HV110 - Asiste a escuela,HV111 - ¿Está viva la madre natural?,HV112 - ID de la madre en el hogar,HV113 - ¿Está vivo el padre natural?,HV114 - Número de orden del padre,HV117 - Elegibilidad para entrevista de mujeres,HV120 - Niños elegibles para medición,HV121 - Asistió a escuela este año,HV122 - Nivel asiste/matriculado actual,HV124 - Años de estudio matriculado,HV125 - Estuvo matriculado año pasado,HV126 - Nivel matriculado año pasado,HV128 - Años de estudio año pasado,HV001 - Número de conglomerado,HV002 - Número de vivienda/hogar,HV004 - Unidad última de muestreo (UPM),hv005 - Factor de ponderación (minúscula),HV008 - Fecha de entrevista (CMC),HV009 - Total de personas en el hogar,HV010 - Mujeres elegibles en el hogar,HV012 - Miembros habituales (De jure),HV013 - Miembros presentes (De facto),HV014 - Niños menores de 5 años,HV015 - Resultado de la entrevista,HV021 - Unidad Primaria de Muestreo (UPM),HV022 - Estrato de la muestra,HV024 - Región,HV025 - Área de residencia,HV026 - Lugar de residencia,HV035 - Niños elegibles para antropometría,HV040 - Altitud del conglomerado en met

### 📉 Análisis de Valores Nulos (Missing Data)

In [14]:
# Calcular porcentaje de nulos por columna
null_percentages = (master.isnull().sum() / len(master) * 100).round(2)
null_df = null_percentages.reset_index()
null_df.columns = ['Variable', '% de Nulos']

# Añadir etiqueta descriptiva
null_df['Descripción'] = null_df['Variable'].apply(lambda x: LABELS.get(x.upper(), 'Sin Etiqueta'))

# Ordenar de mayor a menor porcentaje de nulos
null_df = null_df.sort_values(by='% de Nulos', ascending=False).reset_index(drop=True)

# Reordenar columnas para mejor lectura
null_df = null_df[['Variable', 'Descripción', '% de Nulos']]

# Mostrar el reporte completo (solo las que tienen algún nulo)
display(null_df[null_df['% de Nulos'] > 50])


,Variable,Descripción,% de Nulos
0,QH227B,QH227B - La muestra del agua se extrajo del:,59.27
1,QH227A,QH227A - La muestra fue tomada por:,59.09
2,HV240,HV240 - Tiene chimenea o campana,55.98
3,UBIGEO,UBIGEO - Ubigeo (Código de Ubicación Geográfica),50.06
4,NCONGLOME,NCONGLOME - Número de Conglomerado (mayúscula),50.06
